# CS639: Subproblem-Level Diagnosis of RLVR

Run all three experiments on Google Colab (requires **A100 GPU** runtime).

**Runtime setup:** Runtime → Change runtime type → **A100 GPU**

## 0. Check GPU

In [ ]:
!nvidia-smi

## 1. Clone repo & install dependencies

In [ ]:
# Clone the repo
!git clone https://github.com/jmo25/sparkle.git /content/sparkle
%cd /content/sparkle

In [ ]:
# Install dependencies
!pip install -q torch==2.4.0
!pip install -q vllm datasets matplotlib numpy
!pip install -q pylatexenc sympy antlr4-python3-runtime==4.9.3

# Install VERL (needed for data_loader imports)
%cd /content/sparkle/verl
!pip install -q -e .
%cd /content/sparkle

In [ ]:
# Set vLLM backend
import os
os.environ['VLLM_ATTENTION_BACKEND'] = 'XFORMERS'

## 2. Configure experiment parameters

Adjust these based on your Colab runtime and time budget.

In [ ]:
# ====== Experiment Parameters (modify as needed) ======

MAX_PROBLEMS = 200      # None = all ~6500 problems; start small to test
NUM_SAMPLES = 32        # number of sampled responses per subproblem (for pass@k)
BATCH_SIZE = 32         # vLLM batch size, reduce if OOM

# Which models to run (can comment out to skip)
MODELS_TO_RUN = [
    "base",    # Qwen/Qwen2.5-Math-7B
    "stage1",  # sparkle-reasoning/SparkleRL-7B-Stage1
    "stage2",  # sparkle-reasoning/SparkleRL-7B-Stage2-aug
]

print(f"Problems: {MAX_PROBLEMS or 'all'}")
print(f"Samples per subproblem: {NUM_SAMPLES}")
print(f"Models: {MODELS_TO_RUN}")

## 3. Load & prepare data

In [ ]:
%cd /content/sparkle/experiments

import sys
sys.path.insert(0, '/content/sparkle/experiments')
sys.path.insert(0, '/content/sparkle')

from config import RESULTS_DIR, FIGURES_DIR
from data_loader import load_sparkle_benchmark

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

cache_path = os.path.join(RESULTS_DIR, 'benchmark_data.json')
problems = load_sparkle_benchmark(max_problems=MAX_PROBLEMS, cache_path=cache_path)

print(f"\nLoaded {len(problems)} problems")
print(f"Total subproblem instances: {sum(len(p.subproblems) for p in problems)}")
print(f"\nExample problem: {problems[0].question_raw[:100]}...")
print(f"  Levels: {len(problems[0].subproblems)}, Ground truth: {problems[0].ground_truth}")

## 4. Run inference (most time-consuming step)

Each model is loaded once, runs inference on all subproblems, then unloaded.

Results are cached incrementally — if interrupted, re-running skips completed items.

In [ ]:
from config import MODELS
from inference import run_inference_vllm

for model_name in MODELS_TO_RUN:
    model_path = MODELS[model_name]
    print(f"\n{'='*60}")
    print(f"Model: {model_name} ({model_path})")
    print(f"{'='*60}")
    
    run_inference_vllm(
        model_path=model_path,
        model_name=model_name,
        problems=problems,
        num_samples=NUM_SAMPLES,
        batch_size=BATCH_SIZE,
    )
    
    # Free GPU memory before loading next model
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"\n[{model_name}] Done. GPU memory freed.")

## 5. Experiment 1: Subproblem Pass@k Analysis

**H1:** Does the base model catch up to RL models at high pass@k, even at the subproblem level?

In [ ]:
from exp1_subproblem_passk import run_exp1, find_crossover_points, print_summary_table, save_results

exp1_results = run_exp1(problems)
exp1_crossovers = find_crossover_points(exp1_results)

print_summary_table(exp1_results)

if exp1_crossovers:
    print("\n--- Crossover Analysis ---")
    for comp, levels in exp1_crossovers.items():
        print(f"\n{comp}:")
        for level, data in sorted(levels.items()):
            ck = data['crossover_k']
            print(f"  Level {level}: crossover at k={ck if ck else 'never (RL always better)'}")

save_results(exp1_results, exp1_crossovers, os.path.join(RESULTS_DIR, 'exp1_subproblem_passk.json'))

## 6. Experiment 2: Failure Localization

**H2:** Which reasoning steps are systematically hardest? Does RLVR improve execution or planning?

In [ ]:
from exp2_failure_localization import (
    analyze_failure_localization, compare_failure_patterns, print_failure_summary
)
import json
import numpy as np

exp2_all = {}
for model_name in MODELS_TO_RUN:
    print(f"\n--- Analyzing {model_name} ---")
    result = analyze_failure_localization(problems, model_name, k=1)
    if result:
        exp2_all[model_name] = result

exp2_comparisons = compare_failure_patterns(exp2_all)
print_failure_summary(exp2_all)

# Save
def convert(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

output = {
    'experiment': 'exp2_failure_localization',
    'results': {m: {k: v for k, v in r.items() if k != 'problem_analyses'} for m, r in exp2_all.items()},
    'comparisons': exp2_comparisons,
}
with open(os.path.join(RESULTS_DIR, 'exp2_failure_localization.json'), 'w') as f:
    json.dump(output, f, indent=2, default=convert)
print(f"\nSaved to {RESULTS_DIR}/exp2_failure_localization.json")

## 7. Experiment 3: Path Divergence Analysis

**H3:** When RL models answer correctly but fail subproblems — alternative strategy or shortcut?

In [ ]:
from exp3_path_divergence import (
    analyze_path_divergence, select_manual_review_cases, print_divergence_summary
)

exp3_all = {}
for model_name in MODELS_TO_RUN:
    print(f"\n--- Analyzing {model_name} ---")
    result = analyze_path_divergence(problems, model_name)
    if result:
        exp3_all[model_name] = result

print_divergence_summary(exp3_all)

review_cases = select_manual_review_cases(exp3_all, n_cases=100)
print(f"\nSelected {len(review_cases)} cases for manual review")

# Save
summary = {
    'experiment': 'exp3_path_divergence',
    'results': {m: {k: v for k, v in r.items() if k != 'divergence_cases'} for m, r in exp3_all.items()},
}
with open(os.path.join(RESULTS_DIR, 'exp3_path_divergence.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=convert)
with open(os.path.join(RESULTS_DIR, 'exp3_manual_review.json'), 'w') as f:
    json.dump(review_cases, f, indent=2, default=convert)
print(f"Saved to {RESULTS_DIR}/")

## 8. Generate figures

In [ ]:
from visualize import (
    load_exp_results,
    plot_passk_curves_by_level, plot_passk_curves_by_model, plot_crossover_analysis,
    plot_success_rate_heatmap, plot_transition_gains, plot_first_success_distribution,
    plot_divergence_classification, plot_divergence_rates,
)

# Exp 1 figures
data1 = load_exp_results('exp1_subproblem_passk')
if data1:
    plot_passk_curves_by_level(data1)
    plot_passk_curves_by_model(data1)
    plot_crossover_analysis(data1)

# Exp 2 figures
data2 = load_exp_results('exp2_failure_localization')
if data2:
    plot_success_rate_heatmap(data2)
    plot_transition_gains(data2)
    plot_first_success_distribution(data2)

# Exp 3 figures
data3 = load_exp_results('exp3_path_divergence')
if data3:
    plot_divergence_classification(data3)
    plot_divergence_rates(data3)

print(f"\nAll figures saved to {FIGURES_DIR}/")

## 9. Download results

In [ ]:
# Zip and download all results + figures
!cd /content/sparkle/experiments && zip -r /content/cs639_results.zip results/ figures/

from google.colab import files
files.download('/content/cs639_results.zip')

## (Optional) Mount Google Drive to persist results

Uncomment and run if you want results saved to Drive in case runtime disconnects.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/sparkle/experiments/results /content/drive/MyDrive/cs639_results
# !cp -r /content/sparkle/experiments/figures /content/drive/MyDrive/cs639_figures
# print('Saved to Google Drive')